# **Feature Engineering**

In [20]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

In [21]:
# Load the cleaned dataset
ecommerce_df = pd.read_csv('dataset/cleaned/ecommerce_cleaned.csv', parse_dates=['order_date'])

In [41]:
ecommerce_df.sample(5)

,order_date,order_year,order_month,order_day_of_week,part_day,is_weekend,customer_name,gender,age,age_group,customer_segment,country,order_status,category,sub_category,unit_price_usd,price_tier,quantity_segment,quantity,gross_revenue_usd,discount_amount_usd,discount_percent,discount_tier,is_discounted,order_value_segment,revenue_usd,profit_usd,profit_margin_percent,payment_method,shipping_method,shipping_cost_usd,shipping_cost_ratio,delivery_days,delivery_speed,shipping_country,rating,customer_loyalty_score,loyalty_tier,coupon_used,session_duration_min,engagement_level,pages_visited,abandoned_cart_before,fraud_risk_score,fraud_risk_level,device_type,campaign_source,traffic_source
366840,2024-03-06 00:06:01.539778,2024,3,Wed,Night,No,Chelsea Cook,Female,47,45-54,Regular,United States,Completed,Electronics,Laptops,209.93,Luxury,Small Basket,3,629.79,31.49,5,Low Discount,True,Very High,598.30,169.54,28.34,Credit Card,Express,1.10,0.18,3,Fast,United States,2,3.90,Low,No,53.40,High,5,Yes,88.90,High,Tablet,Google Ads,Search
904436,2025-09-27 01:44:09.582266,2025,9,Sat,Night,Yes,Tim Blair,Male,71,65+,Regular,United States,Completed,Sports,Accessories,223.68,Luxury,Small Basket,2,447.36,67.10,15,Medium Discount,True,High,380.26,112.66,29.63,PayPal,Next Day,19.00,5.00,12,Slow,United States,5,14.10,Low,Yes,52.80,High,6,Yes,42.40,Medium,Mobile,Instagram,Email
207253,2026-01-06 20:23:23.061646,2026,1,Tue,Evening,No,Jennifer Poole,Female,45,45-54,Regular,Italy,Completed,Clothing,Mens Wear,148.50,Premium,Single,1,148.50,0.00,0,No Discount,False,Medium,148.50,46.75,31.48,Apple Pay,Economy,8.61,5.80,9,Slow,Italy,3,28.30,Low,Yes,40.00,Medium,15,Yes,12.90,Low,Mobile,Email,Direct
943868,2025-09-28 17:58:45.351930,2025,9,Sun,Afternoon,Yes,David Hudson,Male,35,35-44,Premium,Germany,Completed,Clothing,Shoes,126.05,Premium,Single,1,126.05,18.91,15,Medium Discount,True,Low,107.14,45.90,42.84,Debit Card,Express,8.10,7.56,11,Slow,Germany,5,89.60,High,Yes,12.70,Low,17,Yes,15.10,Low,Desktop,Affiliate,Referral
601782,2025-07-06 09:02:08.248870,2025,7,Sun,Morning,Yes,Tasha Li,Female,63,55-64,Regular,Canada,Completed,Sports,Sports Wear,28.79,Budget,Bulk,5,143.95,14.39,10,Low Discount,True,Low,129.56,56.81,43.85,Apple Pay,Standard,23.27,17.96,12,Slow,Canada,3,33.00,Low,No,3.30,Low,4,Yes,54.50,Medium,Mobile,Organic,Email


## Time

In [ ]:
# Column for day of the week
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('order_month')+ 1,
    'order_day_of_week',
    ecommerce_df['order_date'].dt.strftime('%a'),
)

ecommerce_df.insert(
    ecommerce_df.columns.get_loc('order_day_of_week') + 1,
    'part_day',
    pd.cut(
        ecommerce_df['order_date'].dt.hour, 
        bins=[0, 6, 12, 18, 24], 
        labels=['Night', 'Morning', 'Afternoon', 'Evening'], 
        right=False
    )
)

## Financial / Revenue

In [ ]:
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('quantity') + 1,
    'gross_revenue_usd',
    ecommerce_df.eval('quantity * unit_price_usd')
)

ecommerce_df.insert(
    ecommerce_df.columns.get_loc('gross_revenue_usd') + 1,
    'discount_amount_usd',
    ecommerce_df.eval('gross_revenue_usd * discount_percent / 100')
)

ecommerce_df.insert(
    ecommerce_df.columns.get_loc('shipping_cost_usd') + 1,
    'shipping_cost_percent',
    ecommerce_df.eval('shipping_cost_usd / revenue_usd * 100')
)

## Discount Features

In [25]:
print(f'Minimum and Maximum Discount Percent: {ecommerce_df["discount_percent"].min()} - {ecommerce_df["discount_percent"].max()}')

ecommerce_df.insert(
    ecommerce_df.columns.get_loc('discount_percent') + 1,
    'discount_tier',
    pd.cut(
        ecommerce_df['discount_percent'], 
        bins=[-1, 0, 10, 20, 25], 
        labels=['No Discount', 'Low Discount', 'Medium Discount', 'High Discount']
    )
)

Minimum and Maximum Discount Percent: 0 - 25


In [26]:
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('discount_tier') + 1,
    'is_discounted',
    ecommerce_df.eval('discount_percent > 0')
)

## Order/Basket Features

In [27]:
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('revenue_usd'),
    'order_value_segment',
    pd.qcut(
        ecommerce_df["revenue_usd"],
        q=4,
        labels=["Low", "Medium", "High", "Very High"]
    )
)

In [28]:
print(f'Minimum and Maximum Quantity {ecommerce_df["quantity"].min()} - {ecommerce_df["quantity"].max()}')

ecommerce_df.insert(
    ecommerce_df.columns.get_loc('quantity'),
    'quantity_segment',
    pd.cut(
        ecommerce_df["quantity"],
        bins=[0, 1, 3, 6],
        labels=["Single", "Small Basket", "Bulk"]
    )
)

Minimum and Maximum Quantity 1 - 5


## Shipping / Logistics Features

In [29]:
print(f'Minimum and Maximum Delivery Days {ecommerce_df["delivery_days"].min()} - {ecommerce_df["delivery_days"].max()}')

ecommerce_df.insert(
    ecommerce_df.columns.get_loc('delivery_days') + 1,
    'delivery_speed',
    pd.cut(
        ecommerce_df["delivery_days"],
        bins=[0, 3, 7, np.inf],
        labels=["Fast", "Standard", "Slow"]
    )
)

Minimum and Maximum Delivery Days 1 - 14


## Customer Features

In [30]:
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('age') + 1,
    'age_group',
    pd.cut(
        ecommerce_df["age"],
        bins=[0, 24, 34, 44, 54, 64, np.inf],
        labels=["18-24", "25-34", "35-44", "45-54", "55-64", "65+"]
    )
)

In [31]:
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('customer_loyalty_score') + 1,
    'loyalty_tier',
    pd.qcut(
        ecommerce_df["customer_loyalty_score"],
        q=3,
        labels=["Low", "Medium", "High"]
    )
)

## Customer Behavior Features

In [32]:
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('session_duration_min') + 1,
    'engagement_level',
    pd.qcut(
        ecommerce_df["session_duration_min"],
        q=3,
        labels=["Low", "Medium", "High"]
    )
)

## Fraud / Risk Features

In [33]:
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('fraud_risk_score') + 1,
    'fraud_risk_level',
    pd.qcut(
        ecommerce_df['fraud_risk_score'],
        q=3,
        labels=["Low", "Medium", "High"],
    )
)

## Product Pricing Features

In [34]:
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('unit_price_usd') + 1,
    'price_tier',
    pd.qcut(
        ecommerce_df["unit_price_usd"],
        q=4,
        labels=["Budget", "Mid-Range", "Premium", "Luxury"]
    )
)

In [40]:
ecommerce_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000123 entries, 0 to 1000122
Data columns (total 48 columns):
 #   Column                  Non-Null Count    Dtype         
---  ------                  --------------    -----         
 0   order_date              1000123 non-null  datetime64[ns]
 1   order_year              1000123 non-null  int64         
 2   order_month             1000123 non-null  int64         
 3   order_day_of_week       1000123 non-null  object        
 4   part_day                1000123 non-null  category      
 5   is_weekend              1000123 non-null  object        
 6   customer_name           1000123 non-null  object        
 7   gender                  1000123 non-null  object        
 8   age                     1000123 non-null  int64         
 9   age_group               1000123 non-null  category      
 10  customer_segment        1000123 non-null  object        
 11  country                 1000123 non-null  object        
 12  order_status  

In [39]:
ecommerce_df.sample(10)

,order_date,order_year,order_month,order_day_of_week,part_day,is_weekend,customer_name,gender,age,age_group,customer_segment,country,order_status,category,sub_category,unit_price_usd,price_tier,quantity_segment,quantity,gross_revenue_usd,discount_amount_usd,discount_percent,discount_tier,is_discounted,order_value_segment,revenue_usd,profit_usd,profit_margin_percent,payment_method,shipping_method,shipping_cost_usd,shipping_cost_ratio,delivery_days,delivery_speed,shipping_country,rating,customer_loyalty_score,loyalty_tier,coupon_used,session_duration_min,engagement_level,pages_visited,abandoned_cart_before,fraud_risk_score,fraud_risk_level,device_type,campaign_source,traffic_source
655505,2025-12-06 15:25:59.758732,2025,12,Sat,Afternoon,Yes,Sergio Pitts,Male,26,25-34,Regular,Germany,Completed,Electronics,Laptops,99.42,Mid-Range,Bulk,5,497.10,74.56,15,Medium Discount,True,High,422.54,103.34,24.46,Bank Transfer,Economy,1.80,0.43,13,Slow,Germany,4,29.90,Low,No,20.50,Low,12,No,41.30,Medium,Tablet,Affiliate,Email
522137,2025-06-12 05:02:31.243702,2025,6,Thu,Night,No,Michael Cardenas,Male,70,65+,VIP,Netherlands,Pending,Health,Supplements,101.67,Mid-Range,Bulk,5,508.35,101.67,20,Medium Discount,True,High,406.68,71.23,17.51,Bank Transfer,Express,22.93,5.64,3,Fast,Netherlands,5,43.90,Medium,Yes,36.70,Medium,17,Yes,64.20,Medium,Desktop,Affiliate,Referral
890385,2024-06-07 15:39:59.296212,2024,6,Fri,Afternoon,No,John Ryan,Male,56,55-64,Regular,United States,Completed,Home,Appliances,37.34,Budget,Small Basket,2,74.68,7.47,10,Low Discount,True,Low,67.21,27.85,41.44,PayPal,Express,2.72,4.05,2,Fast,United States,5,95.00,High,Yes,25.60,Medium,19,Yes,40.10,Medium,Desktop,Facebook,Search
951178,2024-04-18 02:28:19.768563,2024,4,Thu,Night,No,Alexandra Baker,Female,46,45-54,Regular,Canada,Completed,Home,Kitchen,144.60,Premium,Small Basket,2,289.20,0.00,0,No Discount,False,Medium,289.20,145.02,50.15,Apple Pay,Economy,16.25,5.62,14,Slow,Canada,1,85.80,High,No,46.90,High,18,Yes,69.60,High,Mobile,Organic,Direct
308979,2025-07-14 15:00:47.448035,2025,7,Mon,Afternoon,No,Mitchell Thornton,Male,25,25-34,Regular,Italy,Returned,Health,Medical Devices,149.87,Premium,Bulk,5,749.35,74.94,10,Low Discount,True,Very High,674.41,179.11,26.56,Bank Transfer,Economy,18.40,2.73,13,Slow,Italy,3,30.40,Low,No,58.80,High,1,Yes,47.10,Medium,Tablet,Email,Direct
352581,2024-03-02 21:20:29.669627,2024,3,Sat,Evening,Yes,James Foster,Male,30,25-34,Regular,France,Pending,Clothing,Womens Wear,133.97,Premium,Bulk,4,535.88,80.38,15,Medium Discount,True,High,455.50,159.14,34.94,PayPal,Standard,20.45,4.49,12,Slow,France,3,66.40,Medium,Yes,7.80,Low,12,No,65.10,Medium,Mobile,Facebook,Social
758332,2024-07-09 09:12:20.981068,2024,7,Tue,Morning,No,Cynthia Martin,Female,56,55-64,Regular,Germany,Completed,Home,Bedding,228.88,Luxury,Small Basket,3,686.64,0.00,0,No Discount,False,Very High,686.64,330.36,48.11,Debit Card,Next Day,11.05,1.61,13,Slow,Germany,1,49.30,Medium,No,25.50,Medium,10,No,55.40,Medium,Mobile,Facebook,Direct
512152,2025-10-08 12:24:27.180676,2025,10,Wed,Afternoon,No,Christine Navarro,Female,38,35-44,Regular,Australia,Completed,Electronics,Cameras,222.83,Luxury,Small Basket,2,445.66,89.13,20,Medium Discount,True,High,356.53,114.19,32.03,Bank Transfer,Economy,24.11,6.76,1,Fast,Australia,5,63.60,Medium,Yes,20.40,Low,14,No,53.30,Medium,Desktop,Email,Search
503358,2024-04-17 01:08:06.973866,2024,4,Wed,Night,No,Jared Martin,Male,60,55-64,Premium,Italy,Completed,Health,Medical Devices,72.89,Mid-Range,Single,1,72.89,0.00,0,No Discount,False,Low,72.89,29.21,40.07,Debit Card,Economy,23.81,32.67,9,Slow,Italy,2,9.00,Low,Yes,29.70,Medium,2,No,35.50,Medium,Mobile,Email,Referral
919907,2024-06-14 14:38:21.358899,2024,6,Fri,Afternoon,No,Thomas Snow,Male,21,18-24,Regular,Belgium,Completed,Electronics,Tablets,353.32,Luxury,Small Basket,2,706.64,176.66,25,High Discount,True,High,529.98,146.58,27.66,PayPal,Standard,1.56,0.29,4,Standard,Belgium,3,79.40,High,No,13.90,Low,4,No,1.50,Low,Tablet,Ins

## Save Engineered/Processed Dataset

In [ ]:
ecommerce_df.to_csv("dataset/processed/ecommerce_processed.csv")